# Parameter Testing Notebook

This notebook allows you to quickly test different parameters without recompiling Cython:
- Detrending methods
- calc_shape window sizes
- Comet models (comet_curve vs comet_curve2)

Simply modify the functions in the cells below and re-run!

In [ ]:
# Import all necessary libraries
import os
import sys
import math
import warnings
import json
import pandas as pd
import numpy as np
from astropy.io import fits
from astropy.table import Table
from astropy.stats import sigma_clip
from astropy.coordinates import SkyCoord
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
from astropy.timeseries import LombScargle
from matplotlib import pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gs
from wotan import flatten
from scipy.stats import skewnorm

# Add scripts directory for imports
_current_dir = '/Users/azib/Documents/open_source/automated_exocomet_hunt/scripts'
if _current_dir not in sys.path:
    sys.path.insert(0, _current_dir)

# Import the compiled functions you won't modify
from analysis_tools_cython import (
    test_statistic_array,
    single_gaussian_curve_fit,
    skewed_gaussian_curve_fit,
    gauss,
    skewed_gaussian,
    _clean_lightcurve_data
)

warnings.filterwarnings("ignore")
plt.rcParams['agg.path.chunksize'] = 10000

## 🔧 MODIFIABLE FUNCTIONS

### These are the functions you can easily modify:

In [ ]:
# 1. COMET CURVE MODELS - Switch between these easily

def comet_curve(t, A, t0, sigma, tail):
    """
    Calculates the values of an asymmetric Gaussian function representing a comet curve. 
    The difference is the exponential 1/tail term after the mid-transit.
    """
    c = np.ones(len(t))
    ind1 = np.where((t >= (t0 - sigma)) & (t < t0))[0]  # ingress
    ind2 = np.where(t >= t0)[0]  # egress (comet tail)
    
    if len(ind1) > 0:
        c[ind1] = 1 - A * np.exp(-((t[ind1] - t0) ** 2) / (2 * sigma ** 2))
    if len(ind2) > 0:
        c[ind2] = 1 - A * np.exp(-abs(t[ind2] - t0) / tail)
    
    return c

def comet_ingress(x, A, mu, sigma, shape):
    norm = 1 - np.exp(-shape)
    sh = shape / sigma
    return A/norm * (1 - np.exp(-sh * (x - mu + sigma)))

def comet_curve2(x, A, mu, sigma, tail, shape=3):
    """
    Alternative comet curve model with exponential ingress
    MODIFY shape parameter here (currently shape=3)
    """
    return np.piecewise(x, [x<(mu-sigma), np.logical_and(x>=(mu-sigma), x<mu), x>=mu],
                        [0,
                         lambda t: comet_ingress(t, A, mu, sigma, shape),
                         lambda t: A*np.exp(-abs(t-mu)/tail)])

In [ ]:
# 2. COMET CURVE FITTING FUNCTIONS

def comet_curve_fit(x, y):
    """
    Fit an asymmetric comet-like transit curve to data using curve_fit.
    """
    width = (x[-1] - x[0])
    depth = abs(y.min())
    mid = x[len(x)//2]
    
    params_init = [depth, mid, width/8, width/8]
    params_bounds = [[0, x[0], 0, 0], [np.inf, x[-1], width/2, width/2]]
    
    params, cov = curve_fit(comet_curve, x, y, params_init, bounds=params_bounds)
    return params, cov

def comet_curve2_fit(x, y):
    """
    Fit an asymmetric comet-like transit curve with exponential ingress to data using curve_fit.
    """
    width = (x[-1] - x[0])
    depth = abs(y.min())
    mid = x[len(x)//2]
    
    params_init = [depth, mid, width/8, width/8, 3]
    params_bounds = [
        [0, x[0], width/32, width/32, 0.1],
        [np.inf, x[-1], width/2, width/2, 10]
    ]
    
    params, cov = curve_fit(comet_curve2, x, y, params_init, bounds=params_bounds)
    return params, cov

In [ ]:
# 3. SMOOTHING FUNCTION - Modify detrending methods here

def smoothing(table, method, window_length=2.5, power=0.08):
    """
    Smoothing function for lightcurve data.
    
    EASILY CHANGE method parameter or add new methods here!
    """
    time = table[table.colnames[0]]
    flux = table[table.colnames[1]]
    
    # List of available methods - modify as needed
    WOTAN_METHODS = ['biweight', 'lowess', 'median', 'mean', 'rspline', 'hspline', 'trim_mean', 'medfilt']
    LOMBSCARGLE_METHODS = ['lomb-scargle', 'fourier']
    
    if method in WOTAN_METHODS:
        # MODIFY method here - currently uses the input method
        flat_flux, trend_flux = flatten(time, flux, method=method, window_length=window_length)
        return flat_flux, trend_flux
    
    elif method in LOMBSCARGLE_METHODS:
        # Lomb-Scargle detrending
        ls = LombScargle(time, flux)
        frequency, power_ls = ls.autopower()
        
        # MODIFY power threshold here (currently uses input power)
        best_frequency = frequency[power_ls > power]
        
        if len(best_frequency) > 0:
            model = ls.model(time, best_frequency[0])
            flat_flux = flux - model + np.nanmedian(flux)
            return flat_flux, model
        else:
            return flux, np.ones(len(flux))
    
    else:
        # Default: return original flux
        return flux, np.ones(len(flux))

In [ ]:
# 4. CALC_SHAPE FUNCTION - Modify window size and model selection here

def calc_shape(m, n, time, flux, quality, real, flux_error, width, 
               n_m_bg_start=3, n_m_bg_scale_factor=1, comet_model='comet_curve2'):
    """
    Analyse transit shape by fitting symmetric and asymmetric profiles.
    
    KEY PARAMETERS TO MODIFY:
    - width: calc_shape window size
    - comet_model: 'comet_curve' or 'comet_curve2'
    - n_m_bg_start: background window multiplier
    """
    
    # MODIFY: Background window parameters
    n_m_bg = int(width * n_m_bg_start * n_m_bg_scale_factor)
    
    # Extract cutout around transit
    cutout_start = max(0, n - n_m_bg)
    cutout_end = min(len(flux), n + n_m_bg + 1)
    
    t = time[cutout_start:cutout_end]
    x = flux[cutout_start:cutout_end]
    fe = flux_error[cutout_start:cutout_end]
    qu = quality[cutout_start:cutout_end]
    re = real[cutout_start:cutout_end]
    
    if len(t) < 5:  # Need minimum points for fitting
        return 0, 0, 0, 0, 0, 0, [], []
    
    try:
        # Perform Gaussian fitting
        params1, pcov1 = single_gaussian_curve_fit(t, -x)
        
        # CHOOSE MODEL BASED ON PARAMETER
        if comet_model == 'comet_curve':
            params2, pcov2 = comet_curve_fit(t, -x)
            fit2 = -comet_curve(t, *params2)
        elif comet_model == 'comet_curve2':
            params2, pcov2 = comet_curve2_fit(t, -x)
            fit2 = -comet_curve2(t, *params2)
        else:
            raise ValueError(f"Invalid comet_model: {comet_model}. Use 'comet_curve' or 'comet_curve2'")
        
        params3, pcov3 = skewed_gaussian_curve_fit(t, -x, fe, width, params1)
        
        # Generate fits
        fit1 = -gauss(t, *params1)
        fit3 = -skewed_gaussian(t, *params3)
        
        depth = fit3.min()
        
        # Calculate asymmetry metrics
        chi2_sym = np.sum((x - fit1)**2)
        chi2_asym = np.sum((x - fit2)**2)
        chi2_skew = np.sum((x - fit3)**2)
        
        asymmetry = chi2_sym / chi2_asym if chi2_asym > 0 else 1
        amplitude = abs(params2[0]) if len(params2) > 0 else 0
        skewness = params3[3] if len(params3) > 3 else 0
        skewness_error = np.sqrt(np.diag(pcov3))[3] if pcov3.shape[0] > 3 else 0
        
        info = {
            'chi2_sym': chi2_sym,
            'chi2_asym': chi2_asym,
            'chi2_skew': chi2_skew,
            'params_sym': params1,
            'params_asym': params2,
            'params_skew': params3,
            'comet_model_used': comet_model
        }
        
        fits = [fit1, fit2, fit3]
        
        return asymmetry, amplitude, width, skewness, skewness_error, depth, info, fits
        
    except Exception as e:
        print(f"calc_shape fitting failed: {e}")
        return 0, 0, 0, 0, 0, 0, [], []

## 🚀 MAIN PROCESSING FUNCTION

### This is your main processing function - modify parameters at the top:

In [ ]:
# 5. MAIN PROCESSING FUNCTION

def processing(table, f_path='.', lc_info=None, method=None, som_cutouts=False, 
               som_cutouts_directory_name='som_cutouts', make_plots=False, 
               twostep=False, plots_dir='plots/', pipeline=None,
               calc_shape_window_multiplier=1.0, comet_model='comet_curve2'):
    """
    Main processing function - MODIFY PARAMETERS EASILY:
    
    NEW PARAMETERS:
    - calc_shape_window_multiplier: Multiplier for calc_shape window size
    - comet_model: 'comet_curve' or 'comet_curve2'
    """
    
    # === EASILY MODIFIABLE PARAMETERS ===
    MIN_DATA_POINTS = 120  # 2.5 days of data
    DEFAULT_LOMBSCARGLE_POWER = 0.08  # CHANGE THIS
    T_STATISTIC_WINDOW_FACTOR = 60    # CHANGE THIS
    
    # DETRENDING METHOD - CHANGE THIS
    if method is None:
        method = 'median'  # Change default detrending method here
    
    WOTAN_METHODS = ['biweight', 'lowess', 'median', 'mean', 'rspline', 'hspline', 'trim_mean', 'medfilt']
    LOMBSCARGLE_METHODS = ['lomb-scargle', 'fourier']
    # === END MODIFIABLE PARAMETERS ===
    
    original_table = table.copy()
    file_basename = os.path.basename(f_path)
    
    try:
        obj_id = lc_info[0]
    except (TypeError, IndexError):
        obj_id = f_path.split('_')[-1] if '_' in f_path else f_path

    if isinstance(table, pd.DataFrame):
        table = Table.from_pandas(table)
    
    if len(table) > MIN_DATA_POINTS:
        
        # Smoothing operation
        if method in WOTAN_METHODS:
            flat_flux, trend_flux = smoothing(table, method=method)
            table = Table([table[table.colnames[0]], 
                          flat_flux - np.ones(len(flat_flux)), 
                          table[table.colnames[2]], 
                          table[table.colnames[3]]/np.nanmedian(table[table.colnames[1]])],
                         names=('time','flux','quality','flux_error'))
            
        elif method in LOMBSCARGLE_METHODS:
            flat_flux, trend_flux = smoothing(table, method=method, power=DEFAULT_LOMBSCARGLE_POWER)
            table = Table([table[table.colnames[0]], 
                          flat_flux - np.ones(len(flat_flux)), 
                          table[table.colnames[2]], 
                          table[table.colnames[3]]/np.nanmedian(table[table.colnames[1]])],
                         names=('time','flux','quality','flux_error'))

        # Clean the data
        table = _clean_lightcurve_data(table)
        
        if len(table) < MIN_DATA_POINTS:
            return f"{file_basename},INSUFFICIENT_DATA_AFTER_CLEANING", [None, None, None]
        
        # Extract arrays
        t = np.array(table[table.colnames[0]])
        flux = np.array(table[table.colnames[1]])
        quality = np.array(table[table.colnames[2]])
        flux_error = np.array(table[table.colnames[3]])
        real = np.ones(len(flux), dtype=bool)
        
        # Create separate flux for calc_shape
        flux_calc_shape = flux.copy()
        
        # Calculate T-statistic
        T1 = test_statistic_array(flux, T_STATISTIC_WINDOW_FACTOR)
        m, n = np.unravel_index(T1.argmin(), T1.shape)
        
        minT = T1[m, n]
        minT_duration = m + 1
        
        # Apply calc_shape window multiplier
        adjusted_width = minT_duration * calc_shape_window_multiplier
        
        # Run calc_shape with your selected model
        asym, amplitude, width, skewness, skewness_error, depth, info, fits = calc_shape(
            m, n, t, flux_calc_shape, quality, real, flux_error, 
            width=adjusted_width,
            comet_model=comet_model  # EASILY SWITCH MODELS HERE
        )
        
        # Create result string with model info
        result_str = f"{file_basename},model={comet_model},minT={minT:.6f},duration={minT_duration},asym={asym:.4f},depth={depth:.6f}"
        
        # Optional plotting
        if make_plots:
            plt.figure(figsize=(15, 10))
            
            # Full lightcurve
            plt.subplot(2, 3, 1)
            plt.plot(t, flux, '.', markersize=1)
            plt.title('Full Lightcurve (Processed)')
            plt.xlabel('Time')
            plt.ylabel('Flux')
            
            # T-statistic visualization
            plt.subplot(2, 3, 2)
            plt.imshow(T1, aspect='auto', origin='lower')
            plt.colorbar(label='T-statistic')
            plt.plot(n, m, 'r*', markersize=10, label='Minimum T')
            plt.title(f'T-statistic (min={minT:.4f})')
            plt.legend()
            
            # Model fits comparison
            if len(fits) == 3:
                plt.subplot(2, 3, 3)
                cutout_range = slice(max(0, n-int(adjusted_width*2)), min(len(t), n+int(adjusted_width*2)))
                
                plt.plot(t[cutout_range], flux[cutout_range], 'k.', label='Data', markersize=3)
                
                # Only plot fits if they have the right length
                if len(fits[0]) == len(t[cutout_range]):
                    plt.plot(t[cutout_range], fits[0], 'r-', label='Gaussian', linewidth=2)
                    plt.plot(t[cutout_range], fits[1], 'b-', label=f'{comet_model}', linewidth=2)
                    plt.plot(t[cutout_range], fits[2], 'g-', label='Skewed', linewidth=2)
                
                plt.axvline(t[n], color='orange', linestyle='--', alpha=0.7, label='Transit center')
                plt.legend()
                plt.title(f'Model Fits (Model: {comet_model})')
                plt.xlabel('Time')
                plt.ylabel('Flux')
                
                # Chi-squared comparison
                plt.subplot(2, 3, 4)
                models = ['Gaussian', comet_model, 'Skewed']
                chi2_values = [info['chi2_sym'], info['chi2_asym'], info['chi2_skew']]
                colors = ['red', 'blue', 'green']
                plt.bar(models, chi2_values, color=colors, alpha=0.7)
                plt.title('Chi-squared Comparison')
                plt.ylabel('Chi-squared')
                plt.yscale('log')
                
                # Parameters display
                plt.subplot(2, 3, 5)
                param_text = f"""Results:
Asymmetry: {asym:.4f}
Amplitude: {amplitude:.6f}
Depth: {depth:.6f}
Skewness: {skewness:.4f} ± {skewness_error:.4f}
Width: {width:.2f}
Model: {comet_model}

T-statistic: {minT:.6f}
Duration: {minT_duration} points
Window mult: {calc_shape_window_multiplier}
Method: {method}"""
                plt.text(0.1, 0.5, param_text, fontsize=10, verticalalignment='center')
                plt.xlim(0, 1)
                plt.ylim(0, 1)
                plt.axis('off')
                plt.title('Analysis Results')
                
                # Transit zoom
                plt.subplot(2, 3, 6)
                zoom_range = slice(max(0, n-30), min(len(t), n+30))
                plt.plot(t[zoom_range], flux[zoom_range], 'k.', markersize=4)
                plt.axvline(t[n], color='red', linestyle='--', alpha=0.7)
                plt.title('Transit Zoom')
                plt.xlabel('Time')
                plt.ylabel('Flux')
            
            plt.tight_layout()
            plt.show()
        
        return result_str, [t, flux, quality]
    
    else:
        return f"{file_basename},INSUFFICIENT_DATA", [None, None, None]

## 🧪 TESTING SECTION

### Load your data and test different parameters:

In [ ]:
# LOAD YOUR DATA HERE
# Replace with your actual data loading code

# Example:
# table = Table.read('your_lightcurve_file.fits')
# lc_info = ['your_object_id', 'mag', 'sector', etc.]

print("Ready to load your data and test parameters!")
print("\n=== PARAMETER TESTING READY ===")
print("\n📝 TO TEST DIFFERENT PARAMETERS:")
print("\n1. DETRENDING METHODS:")
print("   - In processing(): method='median', 'biweight', 'lowess', 'lomb-scargle'")
print("\n2. CALC_SHAPE WINDOW SIZE:")
print("   - In processing(): calc_shape_window_multiplier=0.5, 1.0, 1.5, 2.0")
print("\n3. COMET MODELS:")
print("   - In processing(): comet_model='comet_curve' or 'comet_curve2'")
print("\n4. COMET_CURVE2 SHAPE PARAMETER:")
print("   - Edit 'shape=3' in comet_curve2() function above")
print("\n🚀 Example usage:")
print("""result, data = processing(table, 
                      method='median',
                      calc_shape_window_multiplier=1.5,
                      comet_model='comet_curve2',
                      make_plots=True)""")

In [ ]:
# QUICK PARAMETER COMPARISON
# Uncomment to run multiple parameter combinations:

# def compare_parameters(table, lc_info=None):
#     """Compare different parameter combinations on the same data"""
#     
#     results = []
#     
#     # Test different combinations
#     test_params = [
#         {'method': 'median', 'comet_model': 'comet_curve', 'calc_shape_window_multiplier': 1.0},
#         {'method': 'median', 'comet_model': 'comet_curve2', 'calc_shape_window_multiplier': 1.0},
#         {'method': 'biweight', 'comet_model': 'comet_curve2', 'calc_shape_window_multiplier': 1.5},
#         {'method': 'lowess', 'comet_model': 'comet_curve2', 'calc_shape_window_multiplier': 0.8},
#     ]
#     
#     for i, params in enumerate(test_params):
#         print(f"\n--- Test {i+1}: {params} ---")
#         result, data = processing(table, lc_info=lc_info, make_plots=False, **params)
#         results.append((params, result))
#         print(result)
#     
#     return results

# Uncomment to run:
# results = compare_parameters(table, lc_info)

print("Uncomment the code above to run parameter comparisons!")

In [ ]:
# TEST YOUR PROCESSING HERE
# Replace with your actual data and test:

# result, data = processing(table, 
#                          f_path='test_file.fits',
#                          lc_info=lc_info,
#                          method='median',              # CHANGE METHOD HERE
#                          comet_model='comet_curve2',   # CHANGE MODEL HERE  
#                          calc_shape_window_multiplier=1.0,  # CHANGE WINDOW SIZE
#                          make_plots=True)              # Set True to see plots
# 
# print(result)

print("Load your data above, then modify parameters and run processing() here!")
print("\n💡 TIP: Start with make_plots=True to see the analysis visually!")